In [ ]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 

df= pd.read_csv(r"D:\Healthcare_AI\dataset\diabetes\diabetes.csv")
print(df.shape)
print(df.columns)
print(df.info())
print(df.describe())
print(df.isnull().sum())

In [3]:
print(df.columns)

Index(['pregnancies', 'glucose', 'bloodpressure', 'skinthickness', 'insulin',
       'bmi', 'diabetespedigreefunction', 'age', 'outcome'],
      dtype='str')


In [ ]:
print("Zeros in each column:")
print((df == 0).sum())

print("\nOutcome Count:")
print(df["outcome"].value_counts())

print("\nOutcome Percentage:")
print(df["outcome"].value_counts(normalize=True) * 100)

In [ ]:
df.hist(figsize=(15,10),bins=20)
plt.tight_layout()
plt.show()

In [ ]:
import seaborn as sns

plt.figure(figsize=(15,10))
for i,column in enumerate(df.columns,1):
    plt.subplot(3,3,i)
    sns.boxplot(df[column])
    plt.title(column)
plt.tight_layout()
plt.show()

In [ ]:
columns=[
    "glucose",
    "bloodpressure",
    "skinthickness",
    "bmi",
    "insulin"
]

df[columns]= df[columns].replace(0,np.nan)
print(df.isnull().sum())

In [ ]:
for column in ["glucose", "bloodpressure", "skinthickness", "insulin", "bmi"]:
    df[column] = df[column].fillna(df[column].median())

print(df.isnull().sum())

In [47]:
df.to_csv("../dataset/diabetes/diabetes_clean.csv", index=False) #../ = move up one folder

In [ ]:
df_clean = pd.read_csv("../dataset/diabetes/diabetes_clean.csv")
print(df_clean.head())
print(df_clean.isnull().sum())

In [49]:
#features
X= df.drop("outcome",axis=1)

#result
y= df["outcome"]

In [50]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test= train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

In [ ]:
print("X_train shape", X_train.shape)
print("X_test shape", X_test.shape)
print("y_train shape", y_train.shape)
print("y_test shape", y_test.shape)

print("Original Dataset")
print(y.value_counts(normalize=True))

print("\nTraining Set")
print(y_train.value_counts(normalize=True))

print("\nTesting Set")
print(y_test.value_counts(normalize=True))

Scaling the data


In [52]:
from sklearn.preprocessing import StandardScaler
scaler= StandardScaler()
X_train= scaler.fit_transform(X_train)
X_test= scaler.transform(X_test)

In [ ]:
print(X_train[:5])
print(X_train.mean(axis=0))
print(X_train.std(axis=0))

In [ ]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(random_state=42)
model.fit(X_train, y_train)

y_pred= model.predict(X_test)
print("actual values:")
print(y_test[:10])
print("\npredicted values:")
print(y_pred[:10])

In [55]:
from sklearn.metrics import accuracy_score
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

Accuracy: 0.7077922077922078


In [56]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)

print(cm)

[[82 18]
 [27 27]]


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1 Score :", f1_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

In [58]:
def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)

    print(f"Accuracy : {accuracy_score(y_test, y_pred):.4f}")
    print(f"Precision: {precision_score(y_test, y_pred):.4f}")
    print(f"Recall   : {recall_score(y_test, y_pred):.4f}")
    print(f"F1 Score : {f1_score(y_test, y_pred):.4f}")
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=5)

knn.fit(X_train, y_train)

evaluate_model(knn, X_test, y_test)

In [ ]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(random_state=42)

dt.fit(X_train, y_train)

evaluate_model(dt, X_test, y_test)

In [61]:
from sklearn.ensemble import RandomForestClassifier
rf= RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)
evaluate_model(rf,X_test, y_test)

Accuracy : 0.7792
Precision: 0.7174
Recall   : 0.6111
F1 Score : 0.6600

Confusion Matrix:
[[87 13]
 [21 33]]

Classification Report:
              precision    recall  f1-score   support

       False       0.81      0.87      0.84       100
        True       0.72      0.61      0.66        54

    accuracy                           0.78       154
   macro avg       0.76      0.74      0.75       154
weighted avg       0.77      0.78      0.77       154



In [62]:
import joblib
joblib.dump(rf, "../models/diabetes_model.pkl")
joblib.dump(scaler, "../models/diabetes_scaler.pkl")

['../models/diabetes_scaler.pkl']

In [ ]:
from sklearn.svm import SVC
svm= SVC(random_state=42)
svm.fit(X_train, y_train)
evaluate_model(svm, X_test, y_test)


Hyperparameter tuning- RandomForest

In [ ]:
from sklearn.model_selection import GridSearchCV
param_grid = {
    "n_estimators":[100,200,300,500],
    "max_depth":[5,10,15,20,None],
    "min_samples_split":[2,5,10,15],
    "min_samples_leaf":[1,2,4,6],
    "max_features":["sqrt","log2",None]
}

grid_search = GridSearchCV(
    estimator= RandomForestClassifier(random_state=42),
    param_grid= param_grid,
    cv=5,
    scoring= "f1",
    n_jobs= -1
)

grid_search.fit(X_train, y_train)

print("Best Parameters:", grid_search.best_params_)
print("Best F1 Score:", grid_search.best_score_)

In [ ]:
best_rf= grid_search.best_estimator_
evaluate_model(best_rf, X_test,y_test)

In [ ]:
import os
print(os.path.exists("../models/diabetes_model.pkl"))

True


In [ ]:
import joblib

loaded_model = joblib.load("../models/diabetes_model.pkl")
loaded_scaler = joblib.load("../models/diabetes_scaler.pkl")

prediction = loaded_model.predict(X_test[:5])

print("Predictions:", prediction)
print("Actual:", y_test.iloc[:5].values)

Predictions: [ True False False False False]
Actual: [False False False  True False]


: 